# R3-S1 follow-up - how strongly do candidate MP3 settings change the audio? (Colab)

The frozen R3-S1 MP3 conditions (64 and 128 kbps at 16 kHz mono) changed validation audio very little. This notebook measures candidate harsher settings
on the **same seeded sample** of Protocol B selection-role recordings (8 each of genuine, MeloTTS, VITS, XTTS), in units comparable with the noise
conditions. CPU runtime is enough. Run all cells top to bottom; safe to rerun after a disconnect.

**Firewall.** Only selection-role validation recordings are read. Test, calibration and operating recordings and ChatterboxTTS are never listed or read. No
model scores anything and no audio is written. The output is signal-level measurement of each candidate against its own original; it decides nothing.

**Produces** in `reports/r3/` on Drive: `r3_s1_mp3_probe_shards.json`, `r3_s1_mp3_probe_shards_rows.csv`, the figure `r3_s1_mp3_probe_shards.png/.svg`,
and the evidence report `r3_s1_mp3_colab_<timestamp>.json`.

In [ ]:
import os, sys, json, subprocess, time, hashlib, platform, shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project"))
assert ROOT.exists(), f"project folder not found: {ROOT}"

SOURCES = {"corruption.py": "src/echoproof/data/corruption.py", "corruption_sample.py": "src/echoproof/data/corruption_sample.py",
           "mp3_probe.py": "src/echoproof/data/mp3_probe.py", "mp3_probe_run.py": "src/echoproof/data/mp3_probe_run.py",
           "measure.py": "src/echoproof/eda/measure.py", "selection.py": "src/echoproof/eda/selection.py", "eda_init.py": "src/echoproof/eda/__init__.py",
           "r2_s1.py": "src/echoproof/features/r2_s1.py", "stats.py": "src/echoproof/features/stats.py", "acoustic.py": "src/echoproof/features/acoustic.py",
           "cohort.py": "src/echoproof/data/cohort.py", "splits.py": "src/echoproof/data/splits.py", "chunks.py": "src/echoproof/data/chunks.py",
           "contracts.py": "src/echoproof/contracts.py", "artifacts.py": "src/echoproof/artifacts.py", "seeding.py": "src/echoproof/training/seeding.py",
           "r3_s1_mp3_probe.py": "scripts/r3_s1_mp3_probe.py", "test_r3_s1_mp3_probe.py": "tests/test_r3_s1_mp3_probe.py",
           "test_r3_s1_corruption.py": "tests/test_r3_s1_corruption.py", "test_r3_s1_sample.py": "tests/test_r3_s1_sample.py",
           "test_r2_s1_features.py": "tests/test_r2_s1_features.py"}
missing = [k for k, rel in SOURCES.items() if not (ROOT / rel).exists()]
assert not missing, f"not found under {ROOT}: {missing} - sync these files to Drive and rerun"
SHA = {k: hashlib.sha256((ROOT / rel).read_bytes()).hexdigest() for k, rel in SOURCES.items()}
print(json.dumps({k: v[:12] for k, v in SHA.items()}, indent=2))

## 0. ffmpeg with the MP3 encoder, environment, frozen inputs

In [ ]:
def ffmpeg_ready():
    exe = shutil.which("ffmpeg")
    if not exe:
        return None
    enc = subprocess.run([exe, "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
    return exe if "libmp3lame" in enc else None

FFMPEG = ffmpeg_ready()
if FFMPEG is None:                                    # Colab normally ships ffmpeg; install only if it is missing
    print(subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], capture_output=True, text=True).stdout[-500:])
    FFMPEG = ffmpeg_ready()
assert FFMPEG, "ffmpeg with libmp3lame is not available on this runtime"
FFMPEG_VERSION = subprocess.run([FFMPEG, "-version"], capture_output=True, text=True).stdout.splitlines()[0]

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["ECHOPROOF_FFMPEG"] = FFMPEG
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))

import numpy, scipy, librosa, pyarrow, pandas, matplotlib
ENV = {"python": platform.python_version(), "numpy": numpy.__version__, "scipy": scipy.__version__, "librosa": librosa.__version__,
       "pyarrow": pyarrow.__version__, "pandas": pandas.__version__, "matplotlib": matplotlib.__version__, "platform": platform.platform(),
       "cpu_count": os.cpu_count(), "ffmpeg": FFMPEG_VERSION}
print(json.dumps(ENV, indent=2))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

INPUTS = {"cohort": ROOT / "data/manifests/cohort_v1.parquet", "split": ROOT / "data/splits/protocol_B_v1.parquet",
          "chunks": ROOT / "data/processed/chunks_v1.parquet", "split_report": ROOT / "reports/r1/split_report.json"}
for name, path in INPUTS.items():
    assert path.exists(), f"missing locked input {name}: {path}"
EXPECTED_COHORT_FILE_SHA256 = "51af27a2dd5c0577bf9c232287ac73b2f66af472ce3308721af83b2e48e8e816"
INPUT_SHA = {name: sha256_file(path) for name, path in INPUTS.items() if name != "split_report"}
assert INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256, "the cohort manifest on Drive is not the frozen GR1 file - stop and compare it with the local copy"
print("cohort manifest is the frozen file")

## 1. Unit tests on this runtime

In [ ]:
proc = subprocess.run([sys.executable, "-m", "pytest", "tests/test_r3_s1_mp3_probe.py", "tests/test_r3_s1_corruption.py", "tests/test_r3_s1_sample.py", "-q", "-p", "no:cacheprovider"],
                      cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(proc.stdout[-3000:] + proc.stderr[-1500:])
UNIT_TESTS_PASSED = proc.returncode == 0
PYTEST_SUMMARY = next((l.strip() for l in reversed(proc.stdout.splitlines()) if " passed" in l or " failed" in l or " error" in l), "no summary line")
print("unit tests:", "PASSED" if UNIT_TESTS_PASSED else "FAILED", "|", PYTEST_SUMMARY)
assert UNIT_TESTS_PASSED, "do not build on a runtime where the tests fail"

## 2. Plan the sample and copy only the shards it needs

In [ ]:
from echoproof.data import corruption, corruption_sample as cs, mp3_probe, mp3_probe_run as pr

PER_COMPONENT, SEED = 8, 42
LOADED = cs.load_inputs(ROOT, SEED)                      # frozen-hash checks, test dropped, Chatterbox rejected, no audio read
SAMPLE = cs.draw_sample(LOADED["eda"].items, PER_COMPONENT, SEED)
needed = sorted({i["shard"] for i in SAMPLE})
SHARD_SRC, SHARD_DIR = ROOT / "data/processed/cohort_v1", Path("/content/shards")
SHARD_DIR.mkdir(parents=True, exist_ok=True)
print(len(SAMPLE), "recordings; shards:", needed)
t0 = time.perf_counter()
for name in needed:
    src, dst = SHARD_SRC / name, SHARD_DIR / name
    assert src.exists(), f"shard missing on Drive: {src}"
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        continue
    for attempt in range(4):
        try:
            tmp = dst.with_name(dst.name + ".part")
            shutil.copyfile(src, tmp)
            assert tmp.stat().st_size == src.stat().st_size, "size differs after copy"
            os.replace(tmp, dst)
            break
        except OSError as exc:
            print(f"  {name}: attempt {attempt + 1} failed ({exc}); retrying")
            time.sleep(5 * 2 ** attempt)
    else:
        raise RuntimeError(f"could not copy {name}")
COPY_SECONDS = round(time.perf_counter() - t0, 1)
print("shards ready |", COPY_SECONDS, "s")

## 3. Run the probe (11 candidates x 32 recordings)

In [ ]:
OUT_DIR = ROOT / "reports/r3"
NAMES = pr.report_names("shards")
if (OUT_DIR / NAMES["report"]).exists():
    print("already built - loading the existing report")
    REPORT = json.loads((OUT_DIR / NAMES["report"]).read_text())
else:
    REPORT = pr.run(ROOT, OUT_DIR, source="shards", shard_dir=SHARD_DIR, per_component=PER_COMPONENT, seed=SEED, ffmpeg=FFMPEG, progress=print, loaded=LOADED)
print(json.dumps(REPORT["sample"], indent=2))
for n, s in REPORT["summary"].items():
    print(f"{n:12s} {s['kbps']:>3} kbps @ {s['encode_rate']:>5} Hz | gain-comp SNR {s['snr_gaincomp_db']['median']:6.2f} dB "
          f"[{s['snr_gaincomp_db']['min']:.1f}, {s['snr_gaincomp_db']['max']:.1f}] | >7k kept {s['retained_7k_pct']['median']:5.1f}% | {s['relative_to_noise']}")

## 4. Independent cross-checks

In [ ]:
import csv, tarfile
import pyarrow.parquet as pq
from echoproof.features.r2_s1 import _read_member

problems = []
rows_path = OUT_DIR / NAMES["rows"]
if sha256_file(rows_path) != REPORT["outputs"][NAMES["rows"]]["sha256"]:
    problems.append("rows file hash differs from the report")
with open(rows_path, newline="") as handle:
    ROWS = list(csv.DictReader(handle))
if len(ROWS) != len(SAMPLE) * len(REPORT["candidates"]):
    problems.append(f"{len(ROWS)} rows for {len(SAMPLE)} recordings x {len(REPORT['candidates'])} candidates")
if any(r["error"] for r in ROWS) or REPORT["errors"]:
    problems.append("some candidate/recording pairs failed")
SPLIT_ROWS = [json.loads(r["payload_json"]) for r in pq.read_table(INPUTS["split"]).to_pylist()]
selection_ids = {r["recording_id"] for r in SPLIT_ROWS if r["partition"] == "validation" and r["validation_role"] == "selection"}
parents = {r["parent_id"] for r in ROWS}
if not parents <= selection_ids:
    problems.append("a parent recording is not a selection-role recording")
if any(":chatterbox:" in p for p in parents):
    problems.append("Chatterbox recordings appear")
if {r["component"] for r in ROWS} != {"genuine", "melo", "vits", "xtts"}:
    problems.append("components differ from genuine/melo/vits/xtts")
if REPORT["sample"]["measured"] != len(SAMPLE) or REPORT["sample"]["unavailable_locally"]:
    problems.append("not every sampled recording was measured")

# Recompute two candidates for three recordings straight from the shards, independently of the run loop.
by_id = {i["recording_id"]: i for i in SAMPLE}
RECOMPUTED = []
for parent in sorted(parents)[::11][:3]:
    item = by_id[parent]
    with tarfile.open(SHARD_DIR / item["shard"]) as tar:
        wave = _read_member(tar, item)
    result = mp3_probe.probe(wave, ["mp3_16k_64", "mp3_16k_16"], FFMPEG)
    for name, metrics in result.items():
        row = next(r for r in ROWS if r["parent_id"] == parent and r["candidate"] == name)
        ok = abs(float(row["snr_gaincomp_db"]) - metrics["snr_gaincomp_db"]) < 1e-6 and int(float(row["lag_samples"])) == metrics["lag_samples"]
        RECOMPUTED.append({"parent": parent, "candidate": name, "match": bool(ok)})
        if not ok:
            problems.append(f"{parent} {name}: recomputed values differ from the table")
print(RECOMPUTED)

# The frozen conditions must read as in the R3-S1 report (same sample, same code).
R3S1 = json.loads(sorted(OUT_DIR.glob("r3_s1_colab_2*.json"))[-1].read_text()) if list(OUT_DIR.glob("r3_s1_colab_2*.json")) else None
print("problems:", problems or "none")

## 5. Verdict and evidence

In [ ]:
checks = {"unit_tests_passed": UNIT_TESTS_PASSED, "cohort_manifest_is_frozen_file": INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256,
          "split_hash_is_frozen_value": REPORT["inputs"]["split_sha256"] == REPORT["inputs"]["split_sha256_frozen_in_report"],
          "all_recordings_measured": REPORT["sample"]["measured"] == len(SAMPLE), "report_errors": REPORT["errors"], "cross_check_problems": problems}
verified = (UNIT_TESTS_PASSED and checks["cohort_manifest_is_frozen_file"] and checks["split_hash_is_frozen_value"] and checks["all_recordings_measured"]
            and not REPORT["errors"] and not problems)
evidence = {"schema_version": 1, "task": "R3-S1", "step": "mp3_probe", "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "environment": ENV, "source_sha256": SHA, "input_sha256": INPUT_SHA,
            "output_sha256": {n: sha256_file(OUT_DIR / n) for n in (NAMES["report"], NAMES["rows"], NAMES["figure"] + ".png", NAMES["figure"] + ".svg")},
            "verdict": {"verified": bool(verified), "pytest_summary": PYTEST_SUMMARY, **checks}, "recomputed": RECOMPUTED, "report": REPORT,
            "timing": {"shard_copy_seconds": COPY_SECONDS}}
path = OUT_DIR / f"r3_s1_mp3_colab_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json"
path.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding="utf-8")
print(json.dumps(evidence["verdict"], indent=2, default=str))
print("\nevidence written to:", path)